# XLM-R on RU22Fact: verdict words masked

Companion to the paper in [MHKhan50/multilingual-factcheck-artefacts](https://github.com/MHKhan50/multilingual-factcheck-artefacts). Same setup as the XLM-R notebook, on artefact-removed evidence from which the verdict words of each language (for example *false*, *fake*, *фейк*, *спростов*, *谣言*) are deleted. If the near-perfect Ukrainian and Russian scores come from reading the verdict, they should fall.

Kaggle settings: **GPU T4**, **Internet on**. Run all cells (about 30 minutes).

In [ ]:
# Settings
REPO_URL = "https://github.com/MHKhan50/multilingual-factcheck-artefacts.git"
REPO_REF = "main"
SEEDS = "0,1,2"
VARIANTS = "cleaned_masked"
OUT = "/kaggle/working/ru22fact-neural-masked"

In [ ]:
import subprocess, sys, torch
print("torch", torch.__version__, "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
import transformers; print("transformers", transformers.__version__)

In [ ]:
# Code and data (RU22Fact is downloaded from its pinned commit and checked by SHA-256)
import os
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "repo"], check=True)
subprocess.run(["git", "clone", "-q", "--branch", REPO_REF, REPO_URL, "repo"], check=True)
os.chdir("/kaggle/working/repo")
print(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "civintel_agentbench.cli", "ru22fact-download", "--data-dir", "data/raw/ru22fact"], check=True)

In [ ]:
# Smoke run (about a minute): catches errors before the long run
subprocess.run([sys.executable, "-m", "civintel_agentbench.cli", "ru22fact-neural", "--data-dir", "data/raw/ru22fact",
                "--output", "/kaggle/working/smoke", "--seeds", "0", "--variants", "cleaned_masked", "--epochs", "1", "--limit", "20"], check=True)

In [ ]:
# Full run: 3 seeds
subprocess.run([sys.executable, "-m", "civintel_agentbench.cli", "ru22fact-neural", "--data-dir", "data/raw/ru22fact",
                "--output", OUT, "--seeds", SEEDS, "--variants", VARIANTS], check=True)

In [ ]:
import json
summary = json.load(open(OUT + "/summary.json"))
for variant, s in summary.items():
    langs = ", ".join("{} {:.3f}".format(k, v["mean"]) for k, v in s["per_language"].items())
    print("{:8s} macro-F1 {:.3f} +/- {:.3f} | {} | {:.0f} min per seed".format(
        variant, s["macro_f1"]["mean"], s["macro_f1"]["std"], langs, s["minutes"]))